# L35 · 欧拉公式遇见 FFT — `e^{-2πikn/N}` 是什么，旋转因子可视化

**今日目标**：用 `cos θ + i·sin θ` 手动拼出 `euler(θ)`，再导出 `twiddle(k, n, N) = euler(-2πkn/N)`。

Week 2 重写 FFT 时每一行都调用 `twiddle`；今天写好这两个函数，下周直接复用。

## 本课剧情：圆周上的坐标表

圆上一个点由角度完全决定：横坐标是 cos θ，纵坐标是 sin θ。欧拉公式把这两个分量合并成一个复数，让旋转变成普通乘法。今天要把这条等式写成 `euler(θ)` 和 `twiddle(k, n, N)` 两个可复用的函数。

## 1. 先把复数当成平面坐标

- 复数 `a + b·i` 对应平面上的点 `(a, b)`。
- 实部 `a` 是横坐标，虚部 `b` 是纵坐标。
- 乘以 `i` 会把点逆时针旋转 90°。
- `e^{iθ}` 是单位圆上角度为 `θ` 的点，也就是 `(cos θ, sin θ)`。

**为什么用 cos+i·sin 手动拼，而不是直接写 `np.exp(1j*theta)`？** 两种写法数值完全相同，但手动拼出实部和虚部可以让你亲眼看到欧拉公式成立——而不只是把它当成黑盒函数调用。Week 2 重写 FFT 时，每对 `(k, n)` 调用一次 `twiddle`，N² 次调用后拼出完整频谱；今天理解底层构造，下周才能看懂每行矩阵在做什么。

## 1.1 复数乘法先看 90° 旋转

先不进入公式，只看 `z * 1j`。每乘一次 `1j`，点就绕原点逆时针转 90°。


In [ ]:
import numpy as np

z = 1 + 0j
for step in range(5):
    print(f'step={step}  z={z.real:+.0f}{z.imag:+.0f}j')
    z = z * 1j


## 实验入口：观察角度序列

这组实验用均匀分布在 [0, 2π) 的角度作为输入，同时对照 cos、sin 和复数坐标，确认三者始终指向单位圆上同一个点。

In [ ]:
import numpy as np
print('就绪；虚数单位示例:', 1j * 1j)  # = (-1+0j)

## 动手观察：从角度序列到复数坐标

调整 `N`（总采样点数）或 `k`（频率编号），观察 `twiddle` 输出的复数点如何在单位圆上重新排列。

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('采样点编号 n =', n)
print('时间轴 t =', np.round(t, 3))
print('角度 angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## 代码实验：角度、三角函数和复数坐标对齐

并排打印同一批角度的 cos、sin 和 exp(1j·θ) 三列，验证 exp(1j·θ) 的实部等于 cos θ、虚部等于 sin θ。

In [ ]:
import numpy as np

angles = np.linspace(0, 2*np.pi, 9)
for theta in angles:
    z = np.exp(1j * theta)
    print(f'theta={theta:5.2f} | cos={z.real:6.3f} | sin={z.imag:6.3f} | z={z.real:6.3f}{z.imag:+6.3f}j')


## 2. ✏️ 任务一：用 cos/sin 手动拼出 `e^{iθ}`

**推理路线**：
1. 欧拉公式 e^{iθ} = cos θ + i·sin θ：实部取 `np.cos(theta)`，虚部取 `np.sin(theta)`。
2. Python 中虚数单位写作 `1j`，虚部项为 `1j * np.sin(theta)`。
3. 两项相加即得复数；NumPy 广播让标量和数组输入都能正常工作。

**参考输入输出**：`euler(0)` = 1+0j；`euler(π/2)` = 0+1j；`euler(-π/2)` = 0-1j

<details><summary>点击查看参考实现</summary>

```python
def euler(theta):
    return np.cos(theta) + 1j * np.sin(theta)
```

</details>

### 写代码前，先把变量表补完整

写 `euler` 前明确三件事：
- 输入：`theta`（弧度，标量或数组）
- 关键步骤：计算 `np.cos(theta)` 作实部，`np.sin(theta)` 作虚部，相加得复数
- 返回：复数（或复数数组），模为 1，辐角为 theta

In [ ]:
def euler(theta):
    # ✏️ TODO: 返回 cosθ + i·sinθ
    return None  # ← 改这里


## 3. 从单位圆到 FFT 旋转因子

`e^{iθ}` 表示“转到角度 θ 的位置”。FFT 需要的是一组规则旋转：

- `N`：一圈分成多少份
- `n`：当前采样点编号
- `k`：当前频率编号
- `-2π·k·n/N`：这个频率在这个采样点要转到的角度

所以 `twiddle(k, n, N)` 只是把这个角度送进欧拉公式。


## 3. ✏️ 任务二：FFT 旋转因子 `twiddle`

`W_N^{kn} = e^{-2πi·k·n/N}`

**推理路线**：
1. 旋转因子的角度是 `-2π·k·n/N`；负号是 DFT 的习惯约定，表示顺时针旋转（分析频率分量时向右转）。
2. 把这个角度送入 `euler(theta)` 即可，不需要重新推导 cos/sin。
3. 优先调用 `euler` 而不是 `np.exp`：Week 2 若要替换底层实现，改一处 `euler` 即可全局生效。

**参考输入输出**：`twiddle(k=1, n=2, N=8)` = `euler(-π/2)` = 0-1j

<details><summary>点击查看参考实现</summary>

```python
def twiddle(k, n, N):
    return euler(-2 * np.pi * k * n / N)
```

</details>

In [ ]:
def twiddle(k, n, N):
    # ✏️ TODO: 返回 e^{-2πi·k·n/N}
    return None  # ← 改这里


## 4. 检查 + 打印单位圆上的点

In [ ]:
theta = np.linspace(0, 2*np.pi, 9)   # 0..360°, 9 个点
z = euler(theta)
assert np.max(np.abs(z - np.exp(1j*theta))) < 1e-12, 'euler 应等于 np.exp(1j*theta)'
assert np.allclose(np.abs(z), 1.0), 'e^{iθ} 模长恒为 1（在单位圆上）'
assert abs(twiddle(0, 0, 8) - 1) < 1e-12, 'k·n=0 时旋转因子应为 1'
for ang, val in zip(np.degrees(theta), z):
    print(f'{ang:6.0f}° | {val.real:+.3f} {val.imag:+.3f}i')
print('\n✅ 复指数=旋转，twiddle=FFT 里的旋转因子。')

## 5. 画出单位圆

In [ ]:
import matplotlib.pyplot as plt
fine = euler(np.linspace(0, 2*np.pi, 200))
plt.figure(figsize=(4.5, 4.5))
plt.plot(fine.real, fine.imag, '-', lw=1)
plt.plot(z.real, z.imag, 'o')
plt.gca().set_aspect('equal')
plt.title('e^{iθ} traces the unit circle')
plt.xlabel('Re'); plt.ylabel('Im'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

**🎉 完成后**：`git commit -m 'learn: Day 4 euler'`（Week 2 重写 FFT 全靠今天）

## 🎨 图示：8 个旋转因子 = FFT 的积木

In [ ]:
import aviz; aviz.style()
aviz.twiddles(8);

In [ ]:
angles = np.linspace(0, 2*np.pi, 5)
for theta in angles:
    z = np.exp(1j*theta)
    reconstructed = np.cos(theta) + 1j*np.sin(theta)
    print(f'theta={theta:.2f} | exp={z:.2f} | cos+i sin={reconstructed:.2f} | match={np.allclose(z, reconstructed)}')


## 参数实验：只改一个旋钮

计算 `np.array([twiddle(1, n, 8) for n in range(8)])` 得到 8 个旋转因子，打印它们的相位（角度），确认分布为 0°, -45°, -90°, -135°, 180°, 135°, 90°, 45°——八个点均匀排列在单位圆上。

把 `N` 从 8 改成 16：相邻旋转因子的角度间距缩小一半（-22.5°），单位圆上的点密度翻倍。把 `k` 从 1 改成 3：每步旋转角度变为 -135°，FFT 频率编号直接控制旋转速度。

In [ ]:
angles = np.linspace(0, 2*np.pi, 5)
for theta in angles:
    z = np.exp(1j*theta)
    reconstructed = np.cos(theta) + 1j*np.sin(theta)
    print(f'theta={theta:.2f} | exp={z:.2f} | cos+i sin={reconstructed:.2f} | match={np.allclose(z, reconstructed)}')


## 本课收束

现在可以调用 `euler(theta)` 生成单位圆上任意角度的复数，也可以调用 `twiddle(k, n, N)` 计算 FFT 所需的旋转因子。这两个函数对应 `src/aurora/audio/dft.py` 里 Week 2 DFT 实现的底层调用。下一节（Day 5）会用窗函数对信号加权，处理后的信号最终会送进今天写好的 `twiddle`。

本节与 `prep_complex_trig/x3_euler.ipynb` 覆盖同一个公式，侧重点不同：x3 建立欧拉公式的数学直觉（单位圆图示、复平面几何），本节把它落地为 FFT 旋转因子的实现。先做过 x3 的话今天的推导会更流畅；没做过的话，完成本节后再回去看 x3 的单位圆图示，两边会相互印证。

In [ ]:
# 小检查：从短序列开始，确认每一步输出
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) N 应该是多少？', N)
print('2) n 是采样点编号：', n)
print('3) t 是每个点的秒数：', np.round(t, 3))
print('4) angle 是每个点在圆上的角度：', np.round(angle, 3))
print('5) x 是最终波形：', np.round(x, 3))
